# Stage 1 Training — Cross-Embodiment Shared Latent Space

Yan et al. (2026) adaptation for dexterous hands.

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- `AIST-hand/` cloned or synced to `MyDrive/AIST-hand/`
- `dex-urdf/` cloned or synced to `MyDrive/dex-urdf/`
- `hograspnet_abl11.csv` in `MyDrive/AIST-hand/grasp-model/data/processed/`

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

Adjust paths if your files live somewhere else in Drive.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/drive/MyDrive/AIST-hand')
DEX_ROOT  = Path('/content/drive/MyDrive/dex-urdf')

# Training hyperparameters — edit for each run
B        = 100_000   # batch size (Yan: 1e5)
N_STEPS  = 5_000     # training steps
LOG_EVERY  = 50      # print losses every N steps
CKPT_EVERY = 500     # save checkpoint every N steps

# Verify files exist
csv  = REPO_ROOT / 'grasp-model/data/processed/hograspnet_abl11.csv'
urdf = DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf'
yaml = REPO_ROOT / 'grasp-model/data/hand_configs/shadow_hand_right.yaml'

for p in [csv, urdf, yaml]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'{status:8s} {p}')

if not all(p.exists() for p in [csv, urdf, yaml]):
    raise FileNotFoundError('Fix missing files before continuing.')

## 3. Install dependencies

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html
!pip install -q torch-geometric
!pip install -q pytorch-kinematics

## 4. Install grasp-model package

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT / 'grasp-model')], check=True)
print('grasp-model installed')

## 5. Run Stage 1 training

In [ ]:
import sys

# Add scripts and cross_emb to path
scripts_dir   = str(REPO_ROOT / 'grasp-model/scripts')
cross_emb_dir = str(REPO_ROOT / 'grasp-model/src/cross_emb')
for p in [scripts_dir, cross_emb_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from pathlib import Path
from cross_embodiment_sampler import CrossEmbodimentSampler
from quat_to_graph import QuatToGraph
from human_modules import HumanEncoder_E_h
from robot_modules import RobotEncoder_E_r, RobotDecoder_D_r
from shared_modules import SharedEncoder_E_X, SharedDecoder_D_X

CSV_PATH    = REPO_ROOT / 'grasp-model/data/processed/hograspnet_abl11.csv'
URDF_PATH   = DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf'
HAND_CONFIG = REPO_ROOT / 'grasp-model/data/hand_configs/shadow_hand_right.yaml'
CKPT_PATH   = REPO_ROOT / 'grasp-model/checkpoints/stage1_latest.pt'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
Z_DIM      = 16
SHARED_DIM = 1024
LR         = 1e-3
LAMBDA_C   = 10.0
LAMBDA_REC = 5.0
LAMBDA_LTC = 1.0
LAMBDA_TMP = 0.1

print(f'Device: {DEVICE}')
print(f'B={B}, N_STEPS={N_STEPS}')

In [ ]:
# Setup sampler and models
sampler = CrossEmbodimentSampler(
    csv_path         = CSV_PATH,
    urdf_path        = URDF_PATH,
    hand_config_path = HAND_CONFIG,
    split            = 'train',
    device           = DEVICE,
)

_probe = sampler.get_batch_temporal(1)
J      = _probe['q_r'].shape[1]
print(f'Robot joints J={J}')

quat_to_graph = QuatToGraph()
E_h = HumanEncoder_E_h(in_dim=4, hidden_dim=32, heads=4, z_dim=Z_DIM).to(DEVICE)
E_r = RobotEncoder_E_r(n_joints=J, shared_dim=SHARED_DIM).to(DEVICE)
E_X = SharedEncoder_E_X(shared_dim=SHARED_DIM, z_dim=Z_DIM).to(DEVICE)
D_X = SharedDecoder_D_X(z_dim=Z_DIM, shared_dim=SHARED_DIM).to(DEVICE)
D_r = RobotDecoder_D_r(n_joints=J, shared_dim=SHARED_DIM).to(DEVICE)

optimizer = torch.optim.Adam(
    list(E_h.parameters()) + list(E_r.parameters()) +
    list(E_X.parameters()) + list(D_X.parameters()) +
    list(D_r.parameters()),
    lr=LR,
)
print('Models ready')

In [ ]:
# Training loop
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)

for step in range(N_STEPS):

    # === SAMPLER ===
    batch          = sampler.get_batch_temporal(B)
    quats_h        = batch['quats_h']
    quats_h_t1     = batch['quats_h_t1']
    q_r            = batch['q_r']
    quats_h_sub    = batch['quats_h_sub']
    quats_r_sub    = batch['quats_r_sub']
    tips_h_sub     = batch['tips_h_sub']
    tips_r_sub     = batch['tips_r_sub']
    tips_h_t1      = batch['tips_h_t1']
    common_fingers = batch['common_fingers']

    # === HUMAN ===
    quats_both = torch.cat([quats_h, quats_h_t1], dim=0)
    graph      = quat_to_graph(quats_both)
    z_both     = E_h(graph.x, graph.edge_index, graph.batch)
    z_t        = z_both[:B]
    z_t1       = z_both[B:]

    # === ROBOT ===
    z_r = E_X(E_r(q_r))

    # === DECODE ===
    q_r_hat    = D_r(D_X(z_r))
    z_h_rt     = E_X(D_X(z_t))
    q_r_hat_t1 = D_r(D_X(z_t1))

    # === LOSSES ===
    L_rec = (q_r - q_r_hat).norm(dim=-1).mean()
    L_ltc = (z_t - z_h_rt).norm(dim=-1).mean()

    z_all    = torch.cat([z_t, z_r], dim=0)
    all_q    = torch.cat([quats_h_sub, quats_r_sub], dim=0)
    all_tips = torch.cat([tips_h_sub.flatten(1), tips_r_sub.flatten(1)], dim=0)
    dot      = (all_q.unsqueeze(1) * all_q.unsqueeze(0)).sum(-1)
    D_R      = (1 - dot ** 2).sum(-1)
    D_ee     = (all_tips.unsqueeze(1) - all_tips.unsqueeze(0)).norm(dim=-1)
    S        = D_R + D_ee
    B2       = z_all.shape[0]
    eye      = torch.eye(B2, dtype=torch.bool, device=DEVICE)
    pos_idx  = S.masked_fill(eye, float('inf')).argmin(dim=1)
    neg_idx  = S.masked_fill(eye, float('-inf')).argmax(dim=1)
    L_cont   = torch.relu((z_all - z_all[pos_idx]).norm(dim=-1) - (z_all - z_all[neg_idx]).norm(dim=-1) + 0.05).mean()

    fk_t   = sampler.robot_rnd.run_fk(q_r_hat)
    fk_t1  = sampler.robot_rnd.run_fk(q_r_hat_t1)
    _, _, meta_t  = sampler.robot_rnd.run_dong_stage2(fk_t,  HAND_CONFIG)
    _, _, meta_t1 = sampler.robot_rnd.run_dong_stage2(fk_t1, HAND_CONFIG)
    tip_labels    = meta_t['tip_labels']
    common_idx_r  = [tip_labels.index(f) for f in common_fingers]
    human_labels  = ['thumb', 'index', 'middle', 'ring', 'pinky']
    common_idx_h  = [human_labels.index(f) for f in common_fingers]
    tips_r_t_sub  = meta_t['tips'].to(DEVICE)[:, common_idx_r, :]
    tips_r_t1_sub = meta_t1['tips'].to(DEVICE)[:, common_idx_r, :]
    tips_h_t1_sub = tips_h_t1[:, common_idx_h, :]
    L_temp = ((tips_h_t1_sub - tips_h_sub) - (tips_r_t1_sub - tips_r_t_sub)).norm(dim=-1).mean()

    L_total = LAMBDA_C * L_cont + LAMBDA_REC * L_rec + LAMBDA_LTC * L_ltc + LAMBDA_TMP * L_temp

    # === BACKWARD ===
    optimizer.zero_grad()
    L_total.backward()
    optimizer.step()

    if step % LOG_EVERY == 0:
        print(f'step {step:05d} | total={L_total.item():.4f} | cont={L_cont.item():.4f} rec={L_rec.item():.4f} ltc={L_ltc.item():.4f} temp={L_temp.item():.4f}')

    if step % CKPT_EVERY == 0:
        torch.save({
            'step':      step,
            'E_h':       E_h.state_dict(),
            'E_r':       E_r.state_dict(),
            'E_X':       E_X.state_dict(),
            'D_X':       D_X.state_dict(),
            'D_r':       D_r.state_dict(),
            'optimizer': optimizer.state_dict(),
        }, CKPT_PATH)

print('Training complete.')